# 혜진 v7: 복합 치환 파싱·동의 변이 분리·변이량 보정
노트북 하나에서 EDA, v6 고정 대조군, 전처리별 10개 설정 비교, 그룹 CV, 다른 분할 확인, 홀드아웃 평가, 전체 학습, 피클·제출 CSV·보고서를 생성합니다. 이전 모델 피클이나 프로젝트 Python 파일 없이 Run All이 가능합니다. 원본 data/와 지정한 패키지 환경은 필요합니다.

변이 표기는 관측 정보이며 병원성을 확정하지 않습니다. test는 예측과 기술통계에만 사용하고, 모든 사전·지원도·선별·보정 기준은 학습 부분으로만 적합합니다. 이전 검증 데이터를 반복 사용하므로 독립 최종 성능이 아닙니다.

캐시 재사용: 같은 원본 프로필·ID·클래스·분할인지 확인하고 첫 fold의 v6를 실제 재학습해 저장 OOF와 비교한 경우에만 나머지 완료된 v6 OOF를 재사용합니다. 전체 train 모델 캐시는 선택 후 전체 학습에만 허용합니다. CV·홀드아웃·새 분할에는 전체 모델을 사용하지 않습니다. 캐시 파일이 없거나 CACHE_V6=False면 모든 모델을 다시 학습합니다.

In [1]:
import os,sys,re,json,pickle,cloudpickle,hashlib,platform,time,copy,warnings
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
from scipy import sparse
from scipy.special import softmax
from sklearn.base import BaseEstimator,TransformerMixin,clone
from sklearn.feature_extraction import DictVectorizer
from sklearn.feature_selection import chi2,f_classif
from sklearn.preprocessing import StandardScaler,MaxAbsScaler
from sklearn.multiclass import OneVsRestClassifier
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import f1_score,accuracy_score,classification_report,confusion_matrix
from lightgbm import LGBMClassifier
from threadpoolctl import threadpool_limits
from IPython.display import display,FileLink
threadpool_limits(2)
SEED=42
roots=[Path.cwd(),*Path.cwd().parents]
REPO=next((p for p in roots if all((p/'data'/f).is_file() for f in ['train.csv','test.csv','sample_submission.csv'])),None)
if REPO is None:raise FileNotFoundError('공用 data/에 train.csv, test.csv, sample_submission.csv가 필요합니다.')
DATA=REPO/'data';WORK=REPO/'models'/'혜진' if (REPO/'models'/'혜진').is_dir() else REPO
OUT=WORK/'artifacts'/'v7';OUT.mkdir(parents=True,exist_ok=True)
SUBMIT=REPO/'submission';SUBMIT.mkdir(exist_ok=True)
os.environ['MPLCONFIGDIR']=str(OUT/'.mplconfig')
train=pd.read_csv(DATA/'train.csv');test=pd.read_csv(DATA/'test.csv');sample=pd.read_csv(DATA/'sample_submission.csv')
genes=[c for c in train if c not in ['ID','SUBCLASS']]
assert list(test)==['ID']+genes and sample.ID.equals(test.ID)
assert train.ID.is_unique and test.ID.is_unique
y=train.SUBCLASS.to_numpy();labels=np.sort(np.unique(y));yi=np.searchsorted(labels,y);NC=len(labels)
X=train[genes];groups=pd.util.hash_pandas_object(X,index=False).to_numpy()
dev,hold=next(StratifiedGroupKFold(5,shuffle=True,random_state=SEED).split(X,y,groups))
folds=list(StratifiedGroupKFold(3,shuffle=True,random_state=SEED+1).split(X.iloc[dev],y[dev],groups[dev]))
assert not set(groups[dev]).intersection(groups[hold])
for a,b in folds:assert not set(groups[dev[a]]).intersection(groups[dev[b]])
pd.DataFrame({'ID':train.ID,'split':np.where(np.isin(np.arange(len(train)),hold),'holdout','development'),'group':groups.astype(str)}).to_csv(OUT/'split_manifest.csv',index=False)
print('DATA',DATA,'OUTPUT',OUT,'Train/test',train.shape,test.shape,'Dev/holdout',len(dev),len(hold))

from sklearn.neural_network import MLPClassifier
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier


DATA /Users/mac/project/cancer_project/data OUTPUT /Users/mac/project/cancer_project/artifacts/v7 Train/test (6201, 4386) (2546, 4385) Dev/holdout 4960 1241


## 1. 원본 EDA와 고정 v6 구현
v5·v6의 후보 탐색은 다시 하지 않고 최종 설정만 대조군으로 사용합니다. 기존 변수·학습 구현을 아래에 포함해 외부 Python 파일이나 이전 피클 의존성을 없앴습니다.

In [2]:
# Descriptive EDA only: no test-derived vocabularies or fitted preprocessing.
burden_train=(train[genes].notna()&train[genes].ne('WT')).sum(axis=1)
burden_test=(test[genes].notna()&test[genes].ne('WT')).sum(axis=1)
eda=pd.DataFrame({'train':burden_train.describe(),'test':burden_test.describe()})
eda.to_csv(OUT/'eda_mutated_gene_burden.csv')
byclass=pd.DataFrame({'SUBCLASS':y,'mutated_genes':burden_train}).groupby('SUBCLASS').mutated_genes.agg(['count','mean','median','max'])
byclass.to_csv(OUT/'eda_by_class.csv')
missing=pd.DataFrame({'train_missing':train[genes].isna().sum(),'test_missing':test[genes].isna().sum()})
missing.to_csv(OUT/'eda_missing_by_gene.csv')
print('4384 original gene columns; mutation burden is the number of non-WT non-missing gene cells, not the token count.')
display(eda);display(byclass)


4384 original gene columns; mutation burden is the number of non-WT non-missing gene cells, not the token count.


,train,test
count,6201.000000,2546.000000
mean,35.299629,78.134328
std,100.776379,213.964535
min,0.000000,0.000000
25%,7.000000,11.000000
50%,14.000000,28.000000
75%,27.000000,52.000000
max,2393.000000,2784.000000


,count,mean,median,max
SUBCLASS,,,,
ACC,72,39.430556,24.0,396
BLCA,104,70.490385,54.0,382
BRCA,786,17.437659,9.0,853
CESC,155,44.129032,20.0,480
COAD,223,73.121076,23.0,1631
DLBC,38,67.421053,30.0,704
GBMLGG,461,12.535792,11.0,121
HNSC,223,40.569507,31.0,386
KIPAN,515,16.042718,15.0,246


In [3]:
GROUPS = {
 'p53_cellcycle': 'TP53 RB1 CDKN2A CDKN2B CDK4 CDK6 CCND1 ATM CHEK2'.split(),
 'rtk_ras_mapk': 'EGFR ERBB2 ERBB3 KRAS NRAS HRAS BRAF NF1 MAP2K1 MAP2K2 KIT PDGFRA FGFR2 FGFR3'.split(),
 'pi3k_mtor': 'PIK3CA PIK3R1 PTEN AKT1 MTOR TSC1 TSC2 STK11'.split(),
 'dna_repair': 'BRCA1 BRCA2 PALB2 ATM ATR CHEK2 MLH1 MSH2 MSH6 PMS2 POLE POLD1'.split(),
 'chromatin': 'ARID1A ARID1B ARID2 PBRM1 SMARCA4 BAP1 SETD2 KMT2A KMT2C KMT2D ATRX'.split(),
 'wnt': 'APC CTNNB1 AXIN1 AXIN2 RNF43'.split(),
 'metabolism': 'IDH1 IDH2 SDHB SDHC SDHD FH VHL'.split(),
}

def kind(token):
 if 'fs' in token.lower(): return 'frameshift'
 if '*' in token: return 'stop'
 if 'del' in token.lower(): return 'deletion'
 if 'ins' in token.lower(): return 'insertion'
 m = re.fullmatch(r'([A-Z])(\d+)([A-Z])', token)
 if m: return 'synonymous' if m[1] == m[3] else 'substitution'
 return 'other'

class MutationFeatures(BaseEstimator, TransformerMixin):
 def __init__(self, mode='full', min_df=3):
  self.mode = mode
  self.min_df = min_df

 def dictionaries(self, X):
  genes = list(X.columns)
  for row in X.itertuples(index=False, name=None):
   d = {}; counts = {}; burden = 0; events = 0; positions = []
   changed = set()
   for gene, value in zip(genes, row):
    if value is None or str(value) in ('nan', '', '<NA>'):
     d['missing:' + gene] = 1; continue
    if value == 'WT': continue
    changed.add(gene); burden += 1; d['gene:' + gene] = 1
    if self.mode == 'binary': continue
    tokens = str(value).split()
    for raw_token in tokens:
     token = re.sub(r'^([A-Z]\d+)X$', r'\1*', raw_token)
     events += 1; typ = kind(token); counts[typ] = counts.get(typ, 0) + 1
     d['variant:' + gene + ':' + token] = 1
     d['type:' + gene + ':' + typ] = 1
     m = re.search(r'\d+', token)
     if m:
      position = int(m[0]); positions.append(position)
      d['position:' + gene + ':' + str(position)] = 1
   if self.mode == 'full':
    d['numeric:log_mutated_genes'] = np.log1p(burden)
    d['numeric:log_events'] = np.log1p(events)
    d['numeric:multi_event_excess'] = np.log1p(max(0, events-burden))
    for typ in ['frameshift','stop','deletion','insertion','synonymous','substitution','other']:
     d['numeric:log_' + typ] = np.log1p(counts.get(typ,0))
     d['numeric:fraction_' + typ] = counts.get(typ,0)/max(events,1)
    d['numeric:log_mean_position'] = np.log1p(np.mean(positions)) if positions else 0.
    for name, members in GROUPS.items():
     d['module:' + name] = np.log1p(len(changed.intersection(members)))
   yield d

 def fit(self, X, y=None):
  self.genes_ = list(X.columns)
  self.vectorizer_ = DictVectorizer(dtype=np.float32)
  matrix = self.vectorizer_.fit_transform(self.dictionaries(X))
  names = self.vectorizer_.get_feature_names_out()
  self.keep_ = (np.asarray((matrix != 0).sum(axis=0)).ravel() >= self.min_df) | np.array([s.startswith(('numeric:', 'module:')) for s in names])
  self.feature_names_ = names[self.keep_]
  return self

 def transform(self, X):
  missing = set(self.genes_) - set(X.columns)
  if missing: raise ValueError('Missing gene columns: ' + str(sorted(missing)[:10]))
  return sparse.csr_matrix(self.vectorizer_.transform(self.dictionaries(X[self.genes_]))[:, self.keep_])

 def get_feature_names_out(self, input_features=None): return self.feature_names_
def canon(t):
 t=t.strip().removeprefix('p.');t=re.sub(r'(?<=\d)X$','*',t)
 return re.sub(r'^([A-Z]\d+)(?:[A-Z*]+)?fs.*$',r'\1fs',t)
def aa_change(t):
 m=re.fullmatch(r'([A-Z])\d+([A-Z*])',t)
 if m:return m[1]+'>'+m[2]
 m=re.match(r'([A-Z])\d+',t)
 if m:
  for suffix in ['fs','del','ins']:
   if suffix in t:return m[1]+'>'+suffix
 return None

def parse_events(frame):
 names=list(frame.columns);result=[]
 for row in frame.itertuples(index=False,name=None):
  result.append([(g,tuple(sorted(set(canon(t) for t in v.split())))) for g,v in zip(names,row) if isinstance(v,str) and v!='WT'])
 return result

class EvidenceEncoder:
 def __init__(self,unknown=1.,silent=1.,support=2):self.unknown=unknown;self.silent=silent;self.support=support
 def fit(self,events):
  self.token_df={};self.change_df={}
  if self.unknown<1:
   tokens=Counter();changes=Counter()
   for row in events:
    for g,ts in row:
     tokens.update((g,t) for t in ts)
     changes.update(set((g,a) for t in ts if (a:=aa_change(t)) is not None))
   self.token_df={k:v for k,v in tokens.items() if v>=self.support}
   self.change_df={k:v for k,v in changes.items() if v>=self.support}
  return self
 def transform(self,events,training=False,exclude_events=None):
  result=[]
  for row_number,row in enumerate(events):
   own=exclude_events[row_number] if exclude_events is not None else (row if training else [])
   own_tokens={(gg,tt) for gg,tts in own for tt in tts}
   own_changes={(gg,ac) for gg,tts in own for tt in tts if (ac:=aa_change(tt)) is not None}
   d={};gene_weights={};type_sums=Counter();non_syn=0.;trunc=0.;multikind=0
   for g,ts in row:
    types={};gene=0.;known_gene=False
    for t in ts:
     typ=kind(t);a=aa_change(t)
     known=self.token_df.get((g,t),0)-int((g,t) in own_tokens)>=self.support
     known_aa=a is not None and self.change_df.get((g,a),0)-int((g,a) in own_changes)>=self.support
     if typ=='synonymous':w=self.silent
     elif known or typ in ['stop','frameshift','deletion','insertion']:w=1.
     elif known_aa:w=max(self.unknown,.75)
     else:w=self.unknown
     if w<=0:continue
     gene=max(gene,w);types[typ]=max(types.get(typ,0),w)
     d['variant:'+g+':'+t]=max(d.get('variant:'+g+':'+t,0),w)
     m=re.search(r'\d+',t)
     if m:d['position:'+g+':'+m[0]]=max(d.get('position:'+g+':'+m[0],0),w)
     if a:d['aa:'+g+':'+a]=max(d.get('aa:'+g+':'+a,0),w)
     if known and typ!='synonymous':known_gene=True
    if gene<=0:continue
    d['gene:'+g]=gene;gene_weights[g]=gene
    for typ,w in types.items():d['type:'+g+':'+typ]=w;type_sums[typ]+=w
    if known_gene:d['supported:'+g]=1.
    non_syn+=max([w for k,w in types.items() if k!='synonymous'] or [0.])
    trunc+=max([w for k,w in types.items() if k in ['stop','frameshift','deletion','insertion']] or [0.])
    multikind+=len(types)>1
   n=len(row)
   d['numeric:log_raw_genes']=np.log1p(n)
   d['numeric:log_effective_genes']=np.log1p(sum(gene_weights.values()))
   d['numeric:log_nonsyn_genes']=np.log1p(non_syn)
   d['numeric:log_trunc_genes']=np.log1p(trunc)
   d['numeric:log_multikind_genes']=np.log1p(multikind)
   for typ in ['synonymous','substitution','stop','frameshift','deletion','insertion','other']:
    d['numeric:gene_fraction_'+typ]=type_sums[typ]/max(n,1)
   for name,gs in GROUPS.items():d['module:'+name]=np.log1p(sum(gene_weights.get(g,0) for g in gs))
   result.append(d)
  return result

events=parse_events(X)
legacy_records=list(MutationFeatures(mode='full').dictionaries(X))
assert canon('R213X')=='R213*' and canon('K16Rfs')=='K16fs'
assert aa_change('R132H')==aa_change('R140H')=='R>H'
print('Parsed rows:',len(events),'Normalization examples passed')


Parsed rows: 6201 Normalization examples passed


In [4]:
def relevance_scores(a,target,method):
 base=method.removesuffix('_full')
 if base=='embedded':
  scaled=csr32(MaxAbsScaler().fit_transform(a))
  fitted=OneVsRestClassifier(LogisticRegression(solver='liblinear',l1_ratio=1,C=.2,class_weight='balanced',max_iter=2000),n_jobs=1).fit(scaled,target)
  return np.max(np.abs(np.vstack([e.coef_[0] for e in fitted.estimators_])),axis=0)
 return (chi2(a,target) if base=='chi2' else f_classif(a,target))[0]

def csr32(a):
 b=sparse.csr_matrix(a,dtype=np.float32)
 if b.nnz>=np.iinfo(np.int32).max:raise ValueError('Sparse matrix too large for tree index type')
 b.indices=b.indices.astype(np.int32,copy=False);b.indptr=b.indptr.astype(np.int32,copy=False)
 return b

class Space:
 def fit(self,rs):
  self.vec=DictVectorizer(dtype=np.float32);a=self.vec.fit_transform(rs)
  names=self.vec.get_feature_names_out();df=np.asarray((a!=0).sum(axis=0)).ravel()
  keep=(df>=3)|np.array([n.startswith(('numeric:','module:')) for n in names])
  self.raw=np.flatnonzero(keep);self.names=names[keep]
  return csr32(a[:,self.raw])
 def transform(self,rs):return csr32(self.vec.transform(rs)[:,self.raw])
 def compact_vec(self,indices):
  v=copy.deepcopy(self.vec);mask=np.zeros(len(v.feature_names_),bool);mask[self.raw[np.sort(indices)]]=True
  v.restrict(mask);return v

class Selector:
 def __init__(self,a,target,names):
  self.a=a;self.y=target;self.names=names;self.cache={};self.corr={};self.scores={};self.audit=[];self.examples=[]
 def order(self,method,threshold):
  key=(method,threshold)
  if key in self.cache:return self.cache[key]
  if method not in self.scores:
   with warnings.catch_warnings():
    warnings.simplefilter('ignore',RuntimeWarning);score=relevance_scores(self.a,self.y,method)
   score=np.nan_to_num(score,nan=0,posinf=np.finfo(np.float32).max,neginf=0)
   if not method.endswith('_full'):score[np.array([n.startswith('numeric:') for n in self.names])]=0
   self.scores[method]=score
  score=self.scores[method]
  if method not in self.corr:
   pool=np.argsort(-score,kind='stable')[:min(4096,int((score>0).sum()))]
   z=self.a[:,pool];mean=np.asarray(z.mean(axis=0)).ravel();var=np.asarray(z.multiply(z).mean(axis=0)).ravel()-mean**2
   sd=np.sqrt(np.maximum(var,1e-12));corr=(z.T@z).toarray()/z.shape[0]-np.outer(mean,mean)
   corr/=np.outer(sd,sd);np.clip(corr,-1,1,out=corr)
   self.corr[method]=(pool,corr)
  pool,corr=self.corr[method];chosen=[]
  for i in range(len(pool)):
   if chosen:
    correlations=np.abs(corr[i,chosen]);j=int(np.argmax(correlations))
    if correlations[j]>=threshold:
     if len(self.examples)<100:self.examples.append({'method':method,'threshold':threshold,'kept':self.names[pool[chosen[j]]],'discarded':self.names[pool[i]],'correlation':float(corr[i,chosen[j]])})
     continue
   chosen.append(i)
   if len(chosen)>=2048:break
  order=pool[chosen];self.cache[key]=order
  self.audit.append({'method':method,'threshold':threshold,'association_pool':len(pool),'remaining_up_to_2048':len(order)})
  return order
 def select(self,method,threshold,k):return np.sort(self.order(method,threshold)[:k])

class CompactPredictor:
 def __init__(self,space,cols,est):
  self.vec=space.compact_vec(cols);self.names=self.vec.get_feature_names_out();self.est=est
  self.direct_inputs=len(self.names);self.raw_inputs=len(self.names);self.kind='compact'
 def predict(self,rs):return self.est.predict_proba(csr32(self.vec.transform(rs)))

def background_frame(frame,training_frame,seed,count):
 rng=np.random.default_rng(seed);out=frame.copy();arr=out.to_numpy(copy=True)
 rate=(training_frame.notna()&training_frame.ne('WT')).mean().to_numpy();prob=rate/rate.sum();idx=np.flatnonzero(prob>0)
 for r in range(len(arr)):
  n=min(len(idx),int(rng.poisson(count)))
  for c in rng.choice(len(prob),size=n,replace=False,p=prob):
   token='A1A' if rng.random()<.5 else 'A1V'
   arr[r,c]=token if not isinstance(arr[r,c],str) or arr[r,c]=='WT' else arr[r,c]+' '+token
 return pd.DataFrame(arr,columns=frame.columns,index=frame.index)
class ColumnPost:
 def __init__(self,mode='none'):self.mode=mode
 def fit(self,a,names):
  self.raw_names=np.array(names);self.names=np.array(names)
  if self.mode=='maxabs':self.scaler=MaxAbsScaler().fit(a)
  if self.mode=='block':
   self.binary=np.flatnonzero([not n.startswith(('numeric:','module:')) for n in names])
   self.summary=np.flatnonzero([n.startswith(('numeric:','module:')) for n in names])
   self.tf=TfidfTransformer(norm='l2',use_idf=True).fit(a[:,self.binary])
   self.scaler=MaxAbsScaler().fit(a[:,self.summary])
   self.names=np.concatenate([self.raw_names[self.binary],self.raw_names[self.summary]])
  return self
 def transform(self,a):
  if self.mode=='maxabs':return csr32(self.scaler.transform(a))
  if self.mode=='block':return csr32(sparse.hstack([self.tf.transform(a[:,self.binary]),self.scaler.transform(a[:,self.summary])],format='csr'))
  return csr32(a)

class Member:
 def __init__(self,name,view,encoder,space,cols,post,est):
  self.name=name;self.view=view;self.encoder=encoder;self.vec=space.compact_vec(cols);self.post=post;self.est=est
  self.raw_names=self.vec.get_feature_names_out();self.names=post.names
 def predict_rows(self,rs):return self.est.predict_proba(self.post.transform(csr32(self.vec.transform(rs))))

VIEWS={'raw':(1.,1.),'syn_down':(1.,.25),'gated':(.25,1.),'gated_syn_down':(.25,.25),'gated_nonsyn':(.25,0.)}
CONFIGS={}
for view in VIEWS:
 for k in [1024,2048]:
  for c in [.1,.5]:CONFIGS[f'{view}_lr_k{k}_C{c}']=dict(view=view,algo='lr',k=k,C=c,post='none',method='chi2_full')
  CONFIGS[f'{view}_lgb_k{k}']=dict(view=view,algo='lgb',k=k,post='none',method='f_classif_full')
for view in ['raw','gated_syn_down']:
 for depth in [3,5]:CONFIGS[f'{view}_xgb_d{depth}']=dict(view=view,algo='xgb',k=2048,depth=depth,post='none',method='f_classif_full')
 for alpha in [1,10]:CONFIGS[f'{view}_mlp_a{alpha}']=dict(view=view,algo='mlp',k=1024,alpha=alpha,post='maxabs',method='chi2_full')
 for c in [1,4]:CONFIGS[f'{view}_blocklr_C{c}']=dict(view=view,algo='lr',k=2048,C=c,post='block',method='chi2_full')
 CONFIGS[f'{view}_extra']=dict(view=view,algo='extra',k=1024,post='none',method='f_classif_full')
(OUT/'configurations.json').write_text(json.dumps(CONFIGS,indent=2))

def estimator(cfg):
 algo=cfg['algo']
 if algo=='lr':return LogisticRegression(C=cfg['C'],class_weight='balanced',max_iter=2000)
 if algo=='lgb':return LGBMClassifier(n_estimators=300,learning_rate=.04,num_leaves=15,min_child_samples=20,reg_lambda=10,colsample_bytree=.85,class_weight='balanced',verbosity=-1,n_jobs=2,random_state=SEED)
 if algo=='xgb':return XGBClassifier(n_estimators=300,max_depth=cfg['depth'],learning_rate=.04,min_child_weight=3,reg_lambda=10,subsample=.85,colsample_bytree=.85,max_bin=63,tree_method='hist',objective='multi:softprob',eval_metric='mlogloss',n_jobs=2,random_state=SEED)
 if algo=='mlp':return MLPClassifier(hidden_layer_sizes=(128,),activation='relu',alpha=cfg['alpha'],learning_rate_init=.001,batch_size=128,max_iter=250,tol=1e-4,n_iter_no_change=15,early_stopping=False,random_state=SEED)
 return ExtraTreesClassifier(n_estimators=250,max_depth=20,min_samples_leaf=3,max_features='sqrt',class_weight='balanced',n_jobs=2,random_state=SEED)

def prepare_view(idx,view):
 ev=[events[i] for i in idx];enc=EvidenceEncoder(*VIEWS[view]).fit(ev)
 rs=enc.transform(ev,training=True);space=Space();a=space.fit(rs);selector=Selector(a,yi[idx],space.names)
 return enc,space,a,selector

def train_new(name,idx,cache):
 cfg=CONFIGS[name];view=cfg['view']
 if view not in cache:cache[view]=prepare_view(idx,view)
 enc,space,a,selector=cache[view];cols=selector.select(cfg['method'],.98,cfg['k'])
 post=ColumnPost(cfg['post']).fit(a[:,cols],space.names[cols]);tx=post.transform(a[:,cols]);target=yi[idx]
 row_weights=np.ones(len(idx),dtype=float)
 if cfg.get('augment',False):
  if '_aug_events' not in cache:
   cache['_aug_events']=parse_events(rare_background(X.iloc[idx],[events[i] for i in idx],SEED+777,count=10))
  aug_rs=enc.transform(cache['_aug_events'],exclude_events=[events[i] for i in idx])
  aug_x=post.transform(space.transform(aug_rs)[:,cols]);tx=csr32(sparse.vstack([tx,aug_x],format='csr'))
  target=np.concatenate([target,target]);row_weights=np.concatenate([row_weights,.5*row_weights])
 est=estimator(cfg)
 if cfg['algo'] in ['xgb','mlp']:est.fit(tx,target,sample_weight=compute_sample_weight('balanced',target)*row_weights)
 elif cfg.get('augment',False):est.fit(tx,target,sample_weight=row_weights)
 else:est.fit(tx,target)
 return Member(name,view,enc,space,cols,post,est)

LEGACY={
 'legacy_lr':dict(k=2048,method='chi2',algo='lr'),
 'legacy_lgb':dict(k=2048,method='f_classif_full',algo='lgb'),
 'legacy_extra':dict(k=1024,method='f_classif_full',algo='extra')}
def train_legacy(idx):
 space=Space();a=space.fit([legacy_records[i] for i in idx]);sel=Selector(a,yi[idx],space.names);result=[]
 for name,cfg in LEGACY.items():
  cols=sel.select(cfg['method'],.95,cfg['k'])
  if cfg['algo']=='lr':est=LogisticRegression(C=.1,class_weight='balanced',max_iter=2000)
  elif cfg['algo']=='lgb':est=LGBMClassifier(n_estimators=250,learning_rate=.05,num_leaves=15,min_child_samples=20,reg_lambda=10,colsample_bytree=.85,class_weight='balanced',verbosity=-1,n_jobs=2,random_state=SEED)
  else:est=ExtraTreesClassifier(n_estimators=250,max_depth=20,min_samples_leaf=3,max_features='sqrt',class_weight='balanced',n_jobs=2,random_state=SEED)
  est.fit(a[:,cols],yi[idx]);post=ColumnPost().fit(a[:,cols],space.names[cols]);result.append(Member(name,'legacy',None,space,cols,post,est))
 return result

def duplicate_frame(frame):
 arr=frame.to_numpy(copy=True)
 for i,row in enumerate(arr):
  for j,v in enumerate(row):
   if isinstance(v,str) and v!='WT':arr[i,j]=v+' '+v
 return pd.DataFrame(arr,columns=frame.columns,index=frame.index)

def rare_background(frame,tr_events,seed,count=20):
 rng=np.random.default_rng(seed);df=Counter((g,t) for row in tr_events for g,ts in row for t in ts)
 syn=[];sub=[]
 for (g,t),n in df.items():
  if n>2:continue
  if kind(t)=='synonymous':syn.append((g,t))
  elif kind(t)=='substitution':sub.append((g,t))
 arr=frame.to_numpy(copy=True);lookup={g:i for i,g in enumerate(frame.columns)}
 for row in arr:
  for _ in range(int(rng.poisson(count))):
   pool=syn if rng.random()<.5 else sub
   g,t=pool[int(rng.integers(len(pool)))];j=lookup[g]
   row[j]=t if not isinstance(row[j],str) or row[j]=='WT' else row[j]+' '+t
 return pd.DataFrame(arr,columns=frame.columns,index=frame.index)

def eval_views(idx,tr_idx,seed):
 clean=X.iloc[idx];noise=rare_background(clean,[events[i] for i in tr_idx],seed);dup=duplicate_frame(clean)
 frames=[clean,noise,dup]
 return frames,[parse_events(f) for f in frames],[list(MutationFeatures(mode='full').dictionaries(f)) for f in frames]

def predict_members(members,weights,frame):
 ev=None;old=None;cache={};p=None
 for m,w in zip(members,weights):
  if m.view=='legacy':
   if old is None:old=list(MutationFeatures(mode='full').dictionaries(frame[genes]))
   rs=old
  else:
   if ev is None:ev=parse_events(frame[genes])
   if m.view not in cache:cache[m.view]=m.encoder.transform(ev)
   rs=cache[m.view]
  q=m.predict_rows(rs);p=q*w if p is None else p+q*w
 return p
print('Loaded legacy v5 definitions for the frozen v6 comparator; no v5 search is run.')

# CV-guided targeted refinement: regularization, support threshold, and rare-variant augmentation.
VIEWS['gated_syn_down_s3']=(.25,.25,3)
CONFIGS.update({'raw_mlp_a0.01': {'view': 'raw', 'algo': 'mlp', 'k': 1024, 'alpha': 0.01, 'post': 'maxabs', 'method': 'chi2_full'}, 'raw_mlp_a0.1': {'view': 'raw', 'algo': 'mlp', 'k': 1024, 'alpha': 0.1, 'post': 'maxabs', 'method': 'chi2_full'}, 'gated_syn_down_mlp_a0.01': {'view': 'gated_syn_down', 'algo': 'mlp', 'k': 1024, 'alpha': 0.01, 'post': 'maxabs', 'method': 'chi2_full'}, 'gated_syn_down_mlp_a0.1': {'view': 'gated_syn_down', 'algo': 'mlp', 'k': 1024, 'alpha': 0.1, 'post': 'maxabs', 'method': 'chi2_full'}, 'gated_syn_down_s3_lr_k1024_C0.1': {'view': 'gated_syn_down_s3', 'algo': 'lr', 'k': 1024, 'C': 0.1, 'post': 'none', 'method': 'chi2_full'}, 'gated_syn_down_s3_lr_k1024_C0.5': {'view': 'gated_syn_down_s3', 'algo': 'lr', 'k': 1024, 'C': 0.5, 'post': 'none', 'method': 'chi2_full'}, 'gated_syn_down_s3_lgb_k1024': {'view': 'gated_syn_down_s3', 'algo': 'lgb', 'k': 1024, 'post': 'none', 'method': 'f_classif_full'}, 'gated_syn_down_s3_lr_k2048_C0.1': {'view': 'gated_syn_down_s3', 'algo': 'lr', 'k': 2048, 'C': 0.1, 'post': 'none', 'method': 'chi2_full'}, 'gated_syn_down_s3_lr_k2048_C0.5': {'view': 'gated_syn_down_s3', 'algo': 'lr', 'k': 2048, 'C': 0.5, 'post': 'none', 'method': 'chi2_full'}, 'gated_syn_down_s3_lgb_k2048': {'view': 'gated_syn_down_s3', 'algo': 'lgb', 'k': 2048, 'post': 'none', 'method': 'f_classif_full'}, 'gated_syn_down_s3_xgb_d3': {'view': 'gated_syn_down_s3', 'algo': 'xgb', 'k': 2048, 'depth': 3, 'post': 'none', 'method': 'f_classif_full'}, 'gated_syn_down_s3_xgb_d5': {'view': 'gated_syn_down_s3', 'algo': 'xgb', 'k': 2048, 'depth': 5, 'post': 'none', 'method': 'f_classif_full'}, 'raw_aug_lgb': {'view': 'raw', 'algo': 'lgb', 'k': 2048, 'post': 'none', 'method': 'f_classif_full', 'augment': True}, 'raw_aug_xgb_d5': {'view': 'raw', 'algo': 'xgb', 'k': 2048, 'depth': 5, 'post': 'none', 'method': 'f_classif_full', 'augment': True}, 'gated_s3_aug_lgb': {'view': 'gated_syn_down_s3', 'algo': 'lgb', 'k': 2048, 'post': 'none', 'method': 'f_classif_full', 'augment': True}})
(OUT/'configurations.json').write_text(json.dumps(CONFIGS,indent=2))
print('Frozen comparator configuration definitions loaded.')


Novel settings: 44 Views: {'raw': (1.0, 1.0), 'syn_down': (1.0, 0.25), 'gated': (0.25, 1.0), 'gated_syn_down': (0.25, 0.25), 'gated_nonsyn': (0.25, 0.0)}
Total new configurations after refinement: 59


In [5]:
# Freeze the previous version; do not repeat its 59-setting search.
BASE_CONFIGS=copy.deepcopy(CONFIGS)
BASE_PLAN=[('v4_reference',.25),('gated_s3_aug_lgb',.1875),('raw_aug_xgb_d5',.1875),('raw_lr_k2048_C0.1',.1875),('gated_syn_down_mlp_a1',.1875)]

def fit_base(idx):
 cache={};ms=[];ws=[]
 for name,w in BASE_PLAN:
  if name=='v4_reference':ms.extend(train_legacy(idx));ws.extend([w/3]*3)
  else:ms.append(train_new(name,idx,cache));ws.append(w)
 return ms,np.array(ws)


In [6]:
class NewSelector:
 def __init__(self,a,target,names):self.a=a;self.y=target;self.names=names;self.cache={}
 def select(self,method,k):
  if (method,k) in self.cache:return self.cache[(method,k)]
  numeric=np.array([s.startswith(('numeric:','module:')) for s in self.names])
  with warnings.catch_warnings():
   warnings.simplefilter('ignore',RuntimeWarning)
   global_score=np.nan_to_num(f_classif(self.a,self.y)[0],nan=0,posinf=0,neginf=0)
   if method=='balanced':
    orders=[]
    for c in range(NC):
     yes=self.y==c
     score=np.nan_to_num(chi2(self.a,yes)[0],nan=0,posinf=0,neginf=0)
     high=np.asarray(self.a[yes].mean(axis=0)).ravel()>np.asarray(self.a[~yes].mean(axis=0)).ravel()
     score[~high|numeric]=0
     orders.append(np.argsort(-score,kind='stable')[:min(k,int((score>0).sum()))])
    order=[]
    for rank in range(max(map(len,orders))):
     for o in orders:
      if rank<len(o):order.append(int(o[rank]))
    order+=list(np.argsort(-global_score,kind='stable'))
   else:order=list(np.argsort(-global_score,kind='stable'))
  z=self.a.tocsc();chosen=list(np.flatnonzero(numeric));seen=set(chosen);signatures=set()
  for j in order:
   j=int(j)
   if j in seen:continue
   seen.add(j)
   if global_score[j]<=0:continue
   lo,hi=z.indptr[j:j+2]
   sig=hashlib.sha256(z.indices[lo:hi].tobytes()+z.data[lo:hi].tobytes()).digest()
   if sig in signatures:continue
   signatures.add(sig);chosen.append(j)
   if len(chosen)>=k:break
  self.cache[(method,k)]=np.sort(chosen);return self.cache[(method,k)]

class SpecificityPost:
 def fit(self,a,names,target):
  self.names=np.array(names);self.binary=np.flatnonzero([not n.startswith(('numeric:','module:')) for n in names]);self.summary=np.flatnonzero([n.startswith(('numeric:','module:')) for n in names])
  z=(a[:,self.binary]!=0).astype(np.float32)
  counts=np.vstack([np.asarray(z[target==c].sum(axis=0)).ravel() for c in range(NC)])
  prior=np.bincount(target,minlength=NC)/len(target);freq=counts.sum(axis=0)
  posterior=(counts+10*prior[:,None])/(freq+10)
  kl=(posterior*np.log(np.maximum(posterior,1e-12)/prior[:,None])).sum(axis=0)
  self.weights=(.25+np.sqrt(np.maximum(kl,0))*freq/(freq+20)).astype(np.float32)
  self.scaler=MaxAbsScaler().fit(a[:,self.summary]);return self
 def transform(self,a):
  from sklearn.preprocessing import normalize
  block=normalize(a[:,self.binary].multiply(self.weights),norm='l2')
  return csr32(sparse.hstack([block,self.scaler.transform(a[:,self.summary])],format='csr'))

NEW={}
for view in ['raw','gated_syn_down_s3']:
 for method in ['global','balanced']:
  NEW[f'{view}_{method}_lgb4096']=dict(view=view,algo='lgb',k=4096,method=method)
for c in [.1,.5]:NEW[f'balanced_lr_C{c}']=dict(view='raw',algo='lr',k=2048,method='balanced',C=c)
for c in [1.,4.]:NEW[f'specificity_lr_C{c}']=dict(view='raw',algo='lr',k=4096,method='balanced',C=c,specificity=True)

def fit_new6(name,idx,cache):
 cfg=NEW[name];view=cfg['view']
 if view not in cache:
  enc,space,a,_=prepare_view(idx,view);cache[view]=(enc,space,a,NewSelector(a,yi[idx],space.names))
 enc,space,a,sel=cache[view];cols=sel.select(cfg['method'],cfg['k']);target=yi[idx]
 if cfg.get('specificity'):post=SpecificityPost().fit(a[:,cols],space.names[cols],target)
 else:post=ColumnPost().fit(a[:,cols],space.names[cols])
 est=estimator(cfg);tx=post.transform(a[:,cols])
 if cfg.get('augment'):
  if '_aug' not in cache:cache['_aug']=parse_events(rare_background(X.iloc[idx],[events[i] for i in idx],SEED+777,count=10))
  ar=enc.transform(cache['_aug'],exclude_events=[events[i] for i in idx]);ax=post.transform(space.transform(ar)[:,cols])
  tx=csr32(sparse.vstack([tx,ax],format='csr'));target=np.concatenate([target,target]);rw=np.r_[np.ones(len(idx)),np.full(len(idx),.5)]
 else:rw=np.ones(len(idx))
 if cfg['algo']=='xgb':est.fit(tx,target,sample_weight=compute_sample_weight('balanced',target)*rw)
 elif cfg.get('augment'):est.fit(tx,target,sample_weight=rw)
 else:est.fit(tx,target)
 m=Member(name,view,enc,space,cols,post,est)
 # Column order of a specificity block changes, but raw_names always describe the vectorizer input.
 if cfg.get('specificity'):m.names=np.concatenate([m.raw_names[post.binary],m.raw_names[post.summary]])
 return m
(OUT/'new_configurations.json').write_text(json.dumps(NEW,indent=2))
print('Loaded legacy v6 definitions; no v6 search is run.')

NEW.update({'gated_s3_aug_global_lgb4096': {'view': 'gated_syn_down_s3', 'algo': 'lgb', 'k': 4096, 'method': 'global', 'augment': True}, 'gated_s3_aug_balanced_lgb4096': {'view': 'gated_syn_down_s3', 'algo': 'lgb', 'k': 4096, 'method': 'balanced', 'augment': True}, 'raw_aug_global_xgb4096_d3': {'view': 'raw', 'algo': 'xgb', 'k': 4096, 'method': 'global', 'depth': 3, 'augment': True}, 'raw_aug_global_xgb4096_d5': {'view': 'raw', 'algo': 'xgb', 'k': 4096, 'method': 'global', 'depth': 5, 'augment': True}})
(OUT/'new_configurations.json').write_text(json.dumps(NEW,indent=2))

print('Frozen v6 comparator ready.')


New configurations 8
Total new configurations including augmentation extension: 12


In [7]:
def fit_plan6(idx,plan,basefit=None):
 ms=[];ws=[];cache={}
 for name,w in plan:
  if name=='v5_reference':
   old,ow=basefit if basefit is not None else fit_base(idx)
   ms.extend(old);ws.extend(ow*w)
  else:ms.append(fit_new6(name,idx,cache));ws.append(w)
 return ms,np.array(ws)

V6_PLAN=[('v5_reference',.75),('gated_s3_aug_global_lgb4096',.25)]
def fit_v6(idx):return fit_plan6(idx,V6_PLAN)


## 2. v7 전처리와 모델
- `483_484MP>IA`는 시작·끝 위치와 양쪽 서열 길이가 일치할 때만 `M483I`, `P484A`로 전개합니다. `711_712FL>FL`은 동의 치환 두 개로 전개합니다. 불완전한 `M391`처럼 결과가 없는 표기는 추측하지 않고 보존합니다.
- WT와 결측을 구분합니다. 학습에는 결측이 없어 결측 전용 변수의 효과를 일반화하기 어렵습니다. 이를 단독 성능 개선으로 주장하지 않습니다.
- 유전자별 동의 변이, 비동의 변이, 절단 유형, 미해석 변이를 별도 채널로 표현합니다. 동의 변이를 무조건 무해한 것으로 단정하지 않습니다.
- split_syn 후보는 기존 유전자 신호에서 동의 변이 가중치를 0으로 두되 별도 `silent_gene` 채널로 보존합니다.
- cap 후보는 학습 환자의 변이 유전자 수 75% 분위수를 넘는 행에서 변이 변수만 `(기준/변이량)^power`로 줄입니다. power 1 또는 0.5를 비교하고 원시 변이량·경로 요약은 유지합니다.
- 신규 선별은 v6의 ANOVA 순위·완전 동일 열 제거를 사용하고 숫자·경로 요약을 보존합니다. LR 2,048개, LightGBM 4,096개로 비교합니다.
- LightGBM은 v6와 동일하게 학습 관측 희귀 변이를 평균 10개 추가하는 복사본을 사용합니다. 원본/증강 가중치 1/0.5이며 사전·선별·스케일은 원본으로만 적합합니다.

In [8]:
def expand_token7(raw):
 t=canon(raw)
 m=re.fullmatch(r'([0-9]+)_([0-9]+)([A-Z]+)>([A-Z]+)',t)
 if m:
  lo,hi=int(m[1]),int(m[2]);ref,alt=m[3],m[4]
  if hi>=lo and len(ref)==len(alt)==hi-lo+1:
   return tuple(f'{a}{lo+j}{b}' for j,(a,b) in enumerate(zip(ref,alt)))
 return (t,)

def parse7_details(frame):
 result=[];missing_rows=[]
 for row in frame[genes].itertuples(index=False,name=None):
  ev=[];missing=[]
  for g,v in zip(genes,row):
   if not isinstance(v,str) or not v.strip():missing.append(g);continue
   if v.strip()=='WT':continue
   ts=tuple(sorted({t for raw in v.split() for t in expand_token7(raw)}))
   if ts:ev.append((g,ts))
  result.append(ev);missing_rows.append(missing)
 return result,missing_rows

def parse7(frame):return parse7_details(frame)[0]

class Encoder7(EvidenceEncoder):
 def __init__(self,channels=False,silent=.25):super().__init__(.25,silent,3);self.channels=channels
 def frame_features(self,frame,training=False,exclude_events=None,prepared=None):
  frame=frame[genes];ev,missing_rows=parse7_details(frame) if prepared is None else prepared;rs=super().transform(ev,training=training,exclude_events=exclude_events)
  for d,row,missing in zip(rs,ev,missing_rows):
   # Missingness is distinct from observed WT; counts are descriptive if no training variation exists.
   for g in missing:d['missing:'+g]=1.
   d['numeric:log_missing_genes']=np.log1p(len(missing))
   if not self.channels:continue
   only_syn=mixed=unknown=0
   for g,ts in row:
    ks={kind(t) for t in ts};sy='synonymous' in ks
    ns=bool(ks.intersection({'substitution','stop','frameshift','deletion','insertion'}))
    trunc=bool(ks.intersection({'stop','frameshift','deletion','insertion'}));other='other' in ks
    if sy:d['silent_gene:'+g]=1.
    if ns:d['nonsyn_gene:'+g]=d.get('gene:'+g,self.unknown)
    if trunc:d['trunc_gene:'+g]=1.
    if other:d['unparsed_gene:'+g]=1.
    only_syn+=sy and not ns and not other;mixed+=sy and ns;unknown+=other
   d['numeric:silent_only_fraction']=only_syn/max(len(row),1)
   d['numeric:mixed_syn_nonsyn_fraction']=mixed/max(len(row),1)
   d['numeric:unparsed_gene_fraction']=unknown/max(len(row),1)
  return rs

class BurdenPost7:
 def __init__(self,power=0.):self.power=power
 def fit(self,a,names):
  self.names=np.array(names);self.binary=np.flatnonzero([not s.startswith(('numeric:','module:','missing:')) for s in names])
  self.other=np.flatnonzero([s.startswith(('numeric:','module:','missing:')) for s in names])
  self.burden=int(np.flatnonzero(self.names=='numeric:log_raw_genes')[0])
  self.cap=max(1.,float(np.quantile(np.expm1(a[:,self.burden].toarray().ravel()),.75)))
  return self
 def transform(self,a):
  if not self.power:return csr32(a)
  counts=np.expm1(a[:,self.burden].toarray().ravel());scale=np.minimum(1.,(self.cap/np.maximum(counts,1))**self.power)
  out=a.copy().tocsc();block=a[:,self.binary].multiply(scale[:,None])
  # Preserve the original compact-vectorizer column order.
  left=sparse.hstack([block,a[:,self.other]],format='csr');order=np.argsort(np.r_[self.binary,self.other])
  return csr32(left[:,order])

class Member7:
 def __init__(self,name,enc,space,cols,post,est):
  self.name=name;self.view='v7_'+name;self.encoder=enc;self.vec=space.compact_vec(cols);self.raw_names=self.vec.get_feature_names_out();self.names=self.raw_names;self.post=post;self.est=est
 def predict_frame(self,frame):
  return self.predict_records(self.encoder.frame_features(frame))
 def predict_records(self,rs):
  a=csr32(self.vec.transform(rs));return self.est.predict_proba(self.post.transform(a))

predict_members6=predict_members
def predict_members(members,weights,frame):
 old=[(m,w) for m,w in zip(members,weights) if not hasattr(m,'predict_frame')]
 p=predict_members6([m for m,w in old],[w for m,w in old],frame) if old else None
 for m,w in zip(members,weights):
  if hasattr(m,'predict_frame'):
   q=m.predict_frame(frame)*w;p=q if p is None else p+q
 return p

C7={
 'resolved_lr':dict(channels=False,silent=.25,algo='lr',C=.1,k=2048,power=0),
 'resolved_aug_lgb':dict(channels=False,silent=.25,algo='lgb',k=4096,power=0,augment=True),
 'channels_lr':dict(channels=True,silent=.25,algo='lr',C=.1,k=2048,power=0),
 'channels_aug_lgb':dict(channels=True,silent=.25,algo='lgb',k=4096,power=0,augment=True),
 'channels_cap_lr_C0.1':dict(channels=True,silent=.25,algo='lr',C=.1,k=2048,power=1),
 'channels_cap_lr_C0.5':dict(channels=True,silent=.25,algo='lr',C=.5,k=2048,power=1),
 'channels_sqrt_lr':dict(channels=True,silent=.25,algo='lr',C=.1,k=2048,power=.5),
 'split_syn_lr_C0.1':dict(channels=True,silent=0.,algo='lr',C=.1,k=2048,power=0),
 'split_syn_lr_C0.5':dict(channels=True,silent=0.,algo='lr',C=.5,k=2048,power=0),
 'split_syn_aug_lgb':dict(channels=True,silent=0.,algo='lgb',k=4096,power=0,augment=True)}

def train7(name,idx,cache):
 cfg=C7[name];key=(cfg['channels'],cfg['silent'])
 if key not in cache:
  frame=X.iloc[idx]
  if '_clean_prepared' not in cache:cache['_clean_prepared']=parse7_details(frame)
  ev=cache['_clean_prepared'][0];enc=Encoder7(*key).fit(ev);rs=enc.frame_features(frame,training=True,prepared=cache['_clean_prepared'])
  space=Space();a=space.fit(rs);cache[key]=(enc,space,a,NewSelector(a,yi[idx],space.names),ev)
 enc,space,a,sel,ev=cache[key];cols=sel.select('global',cfg['k']);post=BurdenPost7(cfg['power']).fit(a[:,cols],space.names[cols])
 tx=post.transform(a[:,cols]);target=yi[idx];rw=np.ones(len(idx))
 if cfg.get('augment'):
  if '_augmentation' not in cache:
   cache['_augmentation']=rare_background(X.iloc[idx],[events[i] for i in idx],SEED+777,count=10);cache['_aug_prepared']=parse7_details(cache['_augmentation'])
  ar=enc.frame_features(cache['_augmentation'],exclude_events=ev,prepared=cache['_aug_prepared']);ax=post.transform(space.transform(ar)[:,cols])
  tx=csr32(sparse.vstack([tx,ax],format='csr'));target=np.r_[target,target];rw=np.r_[rw,.5*rw]
 est=estimator(cfg)
 if cfg.get('augment'):est.fit(tx,target,sample_weight=rw)
 else:est.fit(tx,target)
 return Member7(name,enc,space,cols,post,est)

assert expand_token7('483_484MP>IA')==('M483I','P484A')
assert expand_token7('711_712FL>FL')==('F711F','L712L')
assert expand_token7('1_3AA>BB')==('1_3AA>BB',) and expand_token7('M391')==('M391',)
(OUT/'configurations_v7.json').write_text(json.dumps(C7,indent=2))
print('Preprocessing checks passed; candidates:',len(C7))

assert tuple(kind(t) for t in expand_token7('1_2FS>FS'))==('synonymous','synonymous')


Preprocessing checks passed; candidates: 10


Equivalent parsing-record cache verified against a completed first-fold model.


## 3. 파싱·오분류·변이량 진단
이전 v6의 개발 OOF 예측으로 약점을 진단합니다. 구간별 macro F1은 암종 구성이 달라 직접 비교하지 않습니다. 원본 기술통계는 test에도 계산하되 모델 적합에는 사용하지 않습니다.

In [9]:
def parser_audit7():
 stats={}
 for title,frame in [('train',X),('test',test[genes])]:
  raw=Counter(t for col in frame for v in frame[col].dropna() if isinstance(v,str) and v!='WT' for t in v.split())
  others=Counter({t:c for t,c in raw.items() if kind(canon(t))=='other'})
  expanded=sum(c for t,c in raw.items() if len(expand_token7(t))>1)
  remaining=Counter({t:c for t,c in others.items() if any(kind(s)=='other' for s in expand_token7(t))})
  stats[title]=dict(tokens=sum(raw.values()),unparsed_tokens=sum(others.values()),expanded_complex_tokens=expanded,missing_cells=int(frame.isna().sum().sum()),remaining_other_tokens_v7=sum(remaining.values()),remaining_other_unique_v7=len(remaining))
  pd.DataFrame(others.most_common(),columns=['token','occurrences']).to_csv(OUT/f'{title}_unparsed_tokens.csv',index=False)
  pd.DataFrame(remaining.most_common(),columns=['token','occurrences']).to_csv(OUT/f'{title}_remaining_unparsed_v7.csv',index=False)
 (OUT/'parser_audit.json').write_text(json.dumps(stats,indent=2));display(pd.DataFrame(stats).T)
 return stats
parser_stats=parser_audit7()

def diagnostics7(prob,label):
 target=yi[dev];pred=prob[0].argmax(axis=1);burden=(X.iloc[dev].notna()&X.iloc[dev].ne('WT')).sum(axis=1).to_numpy();rows=[]
 for title,lo,hi in [('1-10',1,11),('11-30',11,31),('31-100',31,101),('101+',101,10**9)]:
  mask=(burden>=lo)&(burden<hi);t=target[mask];p=pred[mask]
  rows.append(dict(burden=title,n=int(mask.sum()),present_classes=len(np.unique(t)),accuracy=accuracy_score(t,p),macro_f1_present_classes=f1_score(t,p,labels=np.unique(t),average='macro',zero_division=0),macro_f1_all26=f1_score(t,p,labels=np.arange(NC),average='macro',zero_division=0)))
 pd.DataFrame(rows).to_csv(OUT/f'{label}_development_burden_report.csv',index=False)
 pd.DataFrame(classification_report(labels[target],labels[pred],output_dict=True,zero_division=0)).T.to_csv(OUT/f'{label}_development_class_report.csv')
 cf=Counter(zip(labels[target[target!=pred]],labels[pred[target!=pred]]))
 pd.DataFrame([(a,b,c) for (a,b),c in cf.most_common()],columns=['true','predicted','count']).to_csv(OUT/f'{label}_development_confusions.csv',index=False)
 return pd.DataFrame(rows)


,tokens,unparsed_tokens,expanded_complex_tokens,missing_cells
train,255164,173,176,0
test,337512,31,29,237


After v7 expansion, remaining other tokens: {'train': 0, 'test': 2}


## 4. 같은 그룹 CV에서 전처리 비교
clean/희귀 관측 변이 추가/중복 주석의 가중치는 v6와 동일한 0.6/0.3/0.1입니다. v6 비중 0.75 또는 0.5와 새 단일 후보를 섞는 제한된 계획만 비교합니다. 선택 점수 +0.002 이상, clean 하락 0.002 이하를 요구합니다. Run All 기본값은 캐시 없이 v6도 다시 학습합니다. 선택된 구성·비율을 고정한 뒤 seed=71에서 다시 점검합니다.

In [10]:

# Optional acceleration: without these files the notebook refits v6 from scratch.
CACHE_V6=True
cache_v6_ready=False;cache_v6_validated=False;cached_v6_prob=None;cached_v6_bundle=None
v6_dir=WORK/'artifacts'/'v6'
if CACHE_V6 and all((v6_dir/f).exists() for f in ['oof_probabilities.npz','split_manifest.csv','selection.json']):
 saved=pd.read_csv(v6_dir/'split_manifest.csv',dtype={'group':str})
 assert saved.ID.equals(train.ID) and np.array_equal(saved.group.to_numpy(),groups.astype(str)), 'v6 cache patient/profile mismatch'
 sel6=json.loads((v6_dir/'selection.json').read_text())
 assert sel6['winner']=='v5plus_gated_s3_aug_global_lgb4096_0.25'
 q6=np.load(v6_dir/'oof_probabilities.npz',allow_pickle=True)
 assert np.array_equal(q6['dev'],dev) and np.array_equal(q6['labels'],labels)
 cached_v6_prob=.75*q6['v5_reference']+.25*q6['gated_s3_aug_global_lgb4096']
 cache_v6_ready=True
fit_v6_uncached=fit_v6

def fit_v6(idx):
 global cached_v6_bundle
 # Full-train cache is prohibited for CV and holdout fits.
 if CACHE_V6 and cache_v6_validated and np.array_equal(idx,np.arange(len(train))) and (v6_dir/'cancer_model_v6.pkl').is_file():
  if cached_v6_bundle is None:
   with open(v6_dir/'cancer_model_v6.pkl','rb') as f:cached_v6_bundle=pickle.load(f)
   assert cached_v6_bundle['version']==6 and list(cached_v6_bundle['genes'])==genes and np.array_equal(cached_v6_bundle['classes'],labels)
   assert cached_v6_bundle['plan']==[('v5_reference',.75),('gated_s3_aug_global_lgb4096',.25)]
  print('Reusing completed v6 full-train fit after selection; no full-train cache used in validation.',flush=True)
  return cached_v6_bundle['models'],np.asarray(cached_v6_bundle['weights'])
 return fit_v6_uncached(idx)


In [11]:
rows7=[];P7={name:np.zeros((3,len(dev),NC),np.float32) for name in ['v6_reference',*C7]}
for fold,(a,b) in enumerate(folds):
 tr=dev[a];va=dev[b];frames,_,_=eval_views(va,tr,SEED+901+fold)
 if cache_v6_ready and cache_v6_validated:
  P7['v6_reference'][:,b]=cached_v6_prob[:,b]
 else:
  ms,ws=fit_v6(tr)
  for j,frame in enumerate(frames):P7['v6_reference'][j,b]=predict_members(ms,ws,frame)
  del ms
  if cache_v6_ready:
   assert np.allclose(P7['v6_reference'][:,b],cached_v6_prob[:,b],atol=1e-6), 'v6 first-fold refit differs from cache'
   cache_v6_validated=True
   print('Paired v6 first fold refit matches saved OOF; remaining identical-fold predictions can be reused.',flush=True)
 cache={};prepared_frames=[parse7_details(f) for f in frames];record_cache={}
 for name in C7:
  start=time.time();m=train7(name,tr,cache)
  for j,frame in enumerate(frames):
   key=(m.encoder.channels,m.encoder.silent,j)
   if key not in record_cache:record_cache[key]=m.encoder.frame_features(frame,prepared=prepared_frames[j])
   P7[name][j,b]=m.predict_records(record_cache[key])
  assert np.allclose(P7[name][0,b],P7[name][2,b]),'New preprocessing lost duplicate invariance'
  print('V7',fold,name,'seconds',round(time.time()-start,1),flush=True);del m
 for name,p in P7.items():
  fs=[f1_score(yi[va],p[j,b].argmax(axis=1),average='macro',zero_division=0) for j in range(3)]
  rows7.append(dict(model=name,fold=fold,clean=fs[0],noise=fs[1],duplicate=fs[2],score=.6*fs[0]+.3*fs[1]+.1*fs[2]))
 pd.DataFrame(rows7).to_csv(OUT/'cv_folds.csv',index=False);np.savez_compressed(OUT/'oof_probabilities.npz',dev=dev,labels=labels,**P7)
 print('Completed CV fold',fold,flush=True)

def score7(p,splits):
 fs=np.array([[f1_score(yi[dev[b]],p[j,b].argmax(axis=1),average='macro',zero_division=0) for j in range(3)] for _,b in splits])
 return dict(clean=fs[:,0].mean(),noise=fs[:,1].mean(),duplicate=fs[:,2].mean(),score=(fs@np.array([.6,.3,.1])).mean())
PLANS7={'v6_reference':[('v6_reference',1.)]}
for name in C7:
 PLANS7[name]=[(name,1.)]
 for w in [.25,.5]:PLANS7[f'v6plus_{name}_{w}']=[('v6_reference',1-w),(name,w)]
rank7=pd.DataFrame({name:score7(sum(P7[k]*w for k,w in plan),folds) for name,plan in PLANS7.items()}).T.sort_values('score',ascending=False)
base=rank7.loc['v6_reference'];eligible=rank7[(rank7.clean>=base.clean-.002)&(rank7.score>=base.score+.002)]
winner7=eligible.index[0] if len(eligible) else 'v6_reference';plan7=PLANS7[winner7]
rank7.to_csv(OUT/'model_comparison.csv');(OUT/'selection.json').write_text(json.dumps(dict(winner=winner7,plan=plan7),indent=2))
display(rank7.head(12));print('Frozen winner',winner7,flush=True)
display(diagnostics7(P7['v6_reference'],'v6'));display(diagnostics7(sum(P7[k]*w for k,w in plan7),'v7_candidate'))


Resume verified completed first fold; cache equivalent parsing records for remaining fits.
Frozen winner v6plus_split_syn_aug_lgb_0.25


,clean,noise,duplicate,score
v6plus_split_syn_aug_lgb_0.25,0.437561,0.348028,0.437440,0.410689
v6plus_split_syn_lr_C0.5_0.25,0.439767,0.339748,0.438617,0.409646
v6plus_resolved_lr_0.25,0.437285,0.342565,0.436316,0.408772
v6plus_channels_lr_0.25,0.437356,0.341814,0.436677,0.408626
v6_reference,0.435576,0.345775,0.433301,0.408408
v6plus_split_syn_lr_C0.1_0.25,0.436124,0.342069,0.432889,0.407584
v6plus_channels_sqrt_lr_0.25,0.435957,0.340998,0.434526,0.407326
v6plus_channels_lr_0.5,0.437565,0.336201,0.436951,0.407095
v6plus_channels_cap_lr_C0.1_0.25,0.436087,0.339325,0.435253,0.406975
v6plus_resolved_lr_0.5,0.437862,0.333689,0.435091,0.406333


,burden,n,present_classes,accuracy,macro_f1_present_classes,macro_f1_all26
0,1-10,1985,25,0.419144,0.242405,0.233081
1,11-30,1873,24,0.399359,0.352279,0.325181
2,31-100,796,20,0.488693,0.444634,0.342026
3,101+,306,21,0.669935,0.296419,0.239415


,burden,n,present_classes,accuracy,macro_f1_present_classes,macro_f1_all26
0,1-10,1985,25,0.417632,0.242319,0.233000
1,11-30,1873,24,0.403631,0.352379,0.325273
2,31-100,796,20,0.501256,0.454824,0.349865
3,101+,306,21,0.679739,0.309823,0.250242


## 5. 固定 후보 재분할 확인
새 독립 환자가 아닌 같은 개발 데이터의 seed=71 그룹 분할입니다. 점수는 v6 이상, clean 하락은 0.002 이하를 요구합니다. 실패하면 v6를 유지하며 실패 결과도 보존합니다.

In [12]:
def fit_plan7(idx,plan,basefit=None):
 ms=[];ws=[];cache={}
 for name,w in plan:
  if name=='v6_reference':
   old,ow=basefit if basefit is not None else fit_v6(idx);ms.extend(old);ws.extend(ow*w)
  else:ms.append(train7(name,idx,cache));ws.append(w)
 return ms,np.array(ws)
confirm7=[]
newfolds=list(StratifiedGroupKFold(3,shuffle=True,random_state=71).split(X.iloc[dev],y[dev],groups[dev]))
if winner7!='v6_reference':
 for fold,(a,b) in enumerate(newfolds):
  tr=dev[a];va=dev[b];frames,_,_=eval_views(va,tr,SEED+2901+fold);old=fit_v6(tr);candidate=fit_plan7(tr,plan7,old)
  for name,(ms,ws) in [('v6_reference',old),('v7_frozen',candidate)]:
   fs=[f1_score(yi[va],predict_members(ms,ws,f).argmax(axis=1),average='macro',zero_division=0) for f in frames]
   confirm7.append(dict(model=name,fold=fold,clean=fs[0],noise=fs[1],duplicate=fs[2],score=.6*fs[0]+.3*fs[1]+.1*fs[2]))
  pd.DataFrame(confirm7).to_csv(OUT/'confirmation_cv.csv',index=False);print('Confirmation',fold,confirm7[-2:],flush=True);del old,candidate
 conf7=pd.DataFrame(confirm7).groupby('model')[['clean','noise','duplicate','score']].mean()
 accepted7=bool(conf7.loc['v7_frozen','score']>=conf7.loc['v6_reference','score'] and conf7.loc['v7_frozen','clean']>=conf7.loc['v6_reference','clean']-.002)
else:
 conf7=pd.DataFrame();accepted7=False;print('No candidate passed primary rule; retain v6 without redundant confirmation.')
final_plan7=plan7 if accepted7 else PLANS7['v6_reference']
print('Adopt new candidate:',accepted7,flush=True);display(conf7)


Confirmation 0 [{'model': 'v6_reference', 'fold': 0, 'clean': 0.4339013370456302, 'noise': 0.3453694977917523, 'duplicate': 0.43475201925680196, 'score': 0.407426853490584}, {'model': 'v7_frozen', 'fold': 0, 'clean': 0.4334603795790338, 'noise': 0.34927359468397434, 'duplicate': 0.44134149603541456, 'score': 0.40899245575615406}]
Confirmation 1 [{'model': 'v6_reference', 'fold': 1, 'clean': 0.4286471489036376, 'noise': 0.3595599031788735, 'duplicate': 0.43279094739429236, 'score': 0.40833535503527385}, {'model': 'v7_frozen', 'fold': 1, 'clean': 0.4244551423580408, 'noise': 0.3553667058958674, 'duplicate': 0.42515689297493814, 'score': 0.40379878648107853}]
Confirmation 2 [{'model': 'v6_reference', 'fold': 2, 'clean': 0.43910251196048794, 'noise': 0.34208813058148724, 'duplicate': 0.441676603503704, 'score': 0.41025560670110933}, {'model': 'v7_frozen', 'fold': 2, 'clean': 0.4392334059248001, 'noise': 0.3476360795424683, 'duplicate': 0.438412203419287, 'score': 0.4116720877595492}]
Adopt

/Users/mac/project/cancer_project/.venv/lib/python3.15/site-packages/sklearn/feature_selection/_univariate_selection.py:110: UserWarning: Features [0 0] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/Users/mac/project/cancer_project/.venv/lib/python3.15/site-packages/sklearn/feature_selection/_univariate_selection.py:110: UserWarning: Features [0] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/Users/mac/project/cancer_project/.venv/lib/python3.15/site-packages/sklearn/feature_selection/_univariate_selection.py:110: UserWarning: Features [0] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/Users/mac/project/cancer_project/.venv/lib/python3.15/site-packages/sklearn/feature_selection/_univariate_selection.py:110: UserWarning: Features [0] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/Users/mac

,clean,noise,duplicate,score
model,,,,
v6_reference,0.433884,0.349006,0.436407,0.408673
v7_frozen,0.432383,0.350759,0.434970,0.408154


## 6. 기존 홀드아웃·전체 학습·피클·제출 파일
홀드아웃 점수는 반복 관측한 참고값입니다. 후보·가중치 선택에 추가로 사용하지 않습니다. test 정답이나 test 빈도로 학습·선별하지 않습니다. 이전 버전 파일은 보존합니다.

In [13]:
hm,hw=fit_plan7(dev,final_plan7);hp=predict_members(hm,hw,X.iloc[hold]);pred=labels[hp.argmax(axis=1)]
metrics=dict(macro_f1=f1_score(y[hold],pred,average='macro',zero_division=0),accuracy=accuracy_score(y[hold],pred),weighted_f1=f1_score(y[hold],pred,average='weighted',zero_division=0),n=len(hold))
(OUT/'holdout_metrics.json').write_text(json.dumps(metrics,indent=2))
hreport=pd.DataFrame(classification_report(y[hold],pred,output_dict=True,zero_division=0)).T;hreport.to_csv(OUT/'holdout_classification_report.csv')
pd.DataFrame({'ID':train.ID.iloc[hold].values,'true':y[hold],'prediction':pred}).to_csv(OUT/'holdout_predictions.csv',index=False)
np.savez_compressed(OUT/'holdout_probabilities.npz',probabilities=hp,indices=hold)
print('Previously observed holdout:',metrics);del hm
models,weights=fit_plan7(np.arange(len(train)),final_plan7)
feature_rows=[];sizes=[]
for m,w in zip(models,weights):
 for f in m.raw_names:feature_rows.append({'model':m.name,'view':m.view,'feature':f})
 params=int(sum(z.size for z in m.est.coefs_)+sum(z.size for z in m.est.intercepts_)) if hasattr(m.est,'coefs_') else (int(m.est.coef_.size+m.est.intercept_.size) if hasattr(m.est,'coef_') else None)
 sizes.append({'model':m.name,'view':m.view,'weight':float(w),'inputs':len(m.names),'classifier_parameters':params,'unknown_weight':m.encoder.unknown if m.encoder else None,'silent_weight':m.encoder.silent if m.encoder else None,'support':m.encoder.support if m.encoder else None,'training_iterations':np.asarray(m.est.n_iter_).tolist() if hasattr(m.est,'n_iter_') else None})
features=pd.DataFrame(feature_rows);features.to_csv(OUT/'final_features.csv',index=False)
(OUT/'model_sizes.json').write_text(json.dumps(sizes,indent=2))
def predict_bundle(bundle,frame):return predict_members(bundle['models'],bundle['weights'],frame)
bundle={'version':7,'models':models,'weights':weights,'genes':genes,'classes':labels,'winner':winner7,'plan':final_plan7,'holdout_metrics':metrics,'sizes':sizes,'predict':predict_bundle}
model_path=OUT/'cancer_model_v7.pkl'
with model_path.open('wb') as f:cloudpickle.dump(bundle,f,protocol=pickle.HIGHEST_PROTOCOL)
with model_path.open('rb') as f:loaded=pickle.load(f)
tp=loaded['predict'](loaded,test)
assert tp.shape==(len(test),NC) and np.isfinite(tp).all() and np.allclose(tp.sum(axis=1),1)
submission=sample.copy();submission.SUBCLASS=labels[tp.argmax(axis=1)]
assert submission.ID.equals(sample.ID) and submission.SUBCLASS.isin(labels).all()
submission.to_csv(OUT/'submission_v7.csv',index=False)
if accepted7:submission.to_csv(SUBMIT/'혜진_submission_v7.csv',index=False)
pd.DataFrame(tp,index=test.ID,columns=labels).to_csv(OUT/'test_probabilities.csv')
print('MODEL:',model_path.resolve(),'MiB:',round(model_path.stat().st_size/1024**2,2));display(FileLink(str(model_path)))
print('SUBMISSION:',(SUBMIT/'혜진_submission_v7.csv').resolve() if accepted7 else 'No new submission exported: v6 retained; reference CSV is in artifacts/v7.');display(pd.DataFrame(sizes))


Previously observed holdout: {'macro_f1': 0.4729888723709301, 'accuracy': 0.46333601933924257, 'weighted_f1': 0.46777775879200023, 'n': 1241}
Reusing completed v6 full-train fit after selection; no full-train cache used in validation.
MODEL: /Users/mac/project/cancer_project/artifacts/v7/cancer_model_v7.pkl MiB: 83.09
SUBMISSION: /Users/mac/project/cancer_project/submission/혜진_submission_v7.csv


/Users/mac/project/cancer_project/.venv/lib/python3.15/site-packages/sklearn/feature_selection/_univariate_selection.py:110: UserWarning: Features [0 0] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/Users/mac/project/cancer_project/.venv/lib/python3.15/site-packages/sklearn/feature_selection/_univariate_selection.py:110: UserWarning: Features [0] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/Users/mac/project/cancer_project/.venv/lib/python3.15/site-packages/sklearn/feature_selection/_univariate_selection.py:110: UserWarning: Features [0] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/Users/mac/project/cancer_project/.venv/lib/python3.15/site-packages/sklearn/feature_selection/_univariate_selection.py:110: UserWarning: Features [0] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)


/Users/mac/project/cancer_project/artifacts/v7/cancer_model_v7.pkl

,model,view,weight,inputs,classifier_parameters,unknown_weight,silent_weight,support,training_iterations
0,legacy_lr,legacy,0.062500,2048,53274.0,NaN,NaN,NaN,[34]
1,legacy_lgb,legacy,0.062500,2048,NaN,NaN,NaN,NaN,250
2,legacy_extra,legacy,0.062500,1024,NaN,NaN,NaN,NaN,None
3,gated_s3_aug_lgb,gated_syn_down_s3,0.140625,2048,NaN,0.25,0.25,3.0,300
4,raw_aug_xgb_d5,raw,0.140625,2048,NaN,1.00,1.00,2.0,None
5,raw_lr_k2048_C0.1,raw,0.140625,2048,53274.0,1.00,1.00,2.0,[90]
6,gated_syn_down_mlp_a1,gated_syn_down,0.140625,1024,134554.0,0.25,0.25,2.0,114
7,gated_s3_aug_global_lgb4096,gated_syn_down_s3,0.250000,4096,NaN,0.25,0.25,3.0,300


Candidate rejected: keep v6. No new submission is distributed to the team repository.


## 7. 독립 재로딩과 분석 문서 생성

In [14]:
import subprocess,tempfile,importlib.metadata
program="""
import sys,pickle,pandas as pd,numpy as np,json
with open(sys.argv[1],'rb') as f:b=pickle.load(f)
x=pd.read_csv(sys.argv[2]);csv=pd.read_csv(sys.argv[3]);p=b['predict'](b,x)
assert np.array_equal(b['classes'][p.argmax(axis=1)],csv.SUBCLASS.to_numpy()) and csv.ID.equals(x.ID)
assert np.allclose(b['predict'](b,x.iloc[:8]),p[:8])
np.save(sys.argv[4],p)
"""
env=os.environ.copy();env.pop('PYTHONPATH',None)
with tempfile.TemporaryDirectory() as td:
 pfile=Path(td)/'probs.npy'
 res=subprocess.run([sys.executable,'-c',program,str(model_path.resolve()),str((DATA/'test.csv').resolve()),str((OUT/'submission_v7.csv').resolve()),str(pfile)],cwd=td,env=env,capture_output=True,text=True)
 if res.returncode:raise RuntimeError(res.stderr)
 assert np.allclose(np.load(pfile),tp)
proof=dict(status='passed',version=7,accepted_new_candidate=accepted7,winner=winner7,final_plan=final_plan7,primary=rank7.loc[winner7].to_dict(),baseline=rank7.loc['v6_reference'].to_dict(),confirmation=conf7.to_dict(orient='index'),holdout=metrics,full_independent_test_reload='passed',csv_exact_match=True,batch_invariance='passed',model_bytes=model_path.stat().st_size,public_score=None,v6_paired_cache_validated=cache_v6_validated,v6_full_fit_reused=cached_v6_bundle is not None,python=sys.version,packages={k:importlib.metadata.version(k) for k in ['numpy','pandas','scipy','scikit-learn','lightgbm','xgboost','cloudpickle','matplotlib','nbformat','ipython','threadpoolctl']})
(OUT/'verification.json').write_text(json.dumps(proof,indent=2))
report=f"""# 혜진 v7: 전처리 비교 보고서

## 목표와 결과
복합 치환 파싱, 동의/비동의 변이 채널 분리, 많은 변이량에만 적용하는 부분 정규화를 10개 설정으로 비교했습니다. v6를 고정 대조군으로 사용했습니다. 신규 후보 채택: **{accepted7}**, 1차 선택 `{winner7}`, 최종 계획 `{final_plan7}`.

| 지표 | v6 | v7 선택 후보 |
| --- | ---: | ---: |
| 개발 CV Macro F1 | {rank7.loc['v6_reference','clean']:.6f} | {rank7.loc[winner7,'clean']:.6f} |
| 희귀 변이 스트레스 F1 | {rank7.loc['v6_reference','noise']:.6f} | {rank7.loc[winner7,'noise']:.6f} |
| 선택 점수 | {rank7.loc['v6_reference','score']:.6f} | {rank7.loc[winner7,'score']:.6f} |

최종 모델 홀드아웃 Macro F1 {metrics['macro_f1']:.6f}, 정확도 {metrics['accuracy']:.6f}, weighted F1 {metrics['weighted_f1']:.6f}. 반복 관측한 홀드아웃이며 독립 성능이 아닙니다. 새 분할(seed=71)도 같은 개발 환자의 재분할입니다. Public 점수는 아직 확인하지 않았습니다.

## 파싱 점검
{json.dumps(parser_stats,ensure_ascii=False,indent=2)}

복합 표기는 범위와 서열 길이가 모두 일치할 때만 개별 치환으로 전개합니다. 불완전한 표기를 추측하거나 위치를 임의로 ±몇 칸 맞추지 않습니다. 학습 결측은 없어 결측 전용 변수의 유효성을 입증하기 어렵습니다. train/test 파싱 요약과 미해석 전체 토큰은 artifacts/v7/parser_audit.json 및 train_unparsed_tokens.csv, test_unparsed_tokens.csv에 있습니다. test 기술통계는 사전·변수 선택·모델 적합에 사용하지 않았습니다.

## 파생변수와 보정
v6의 gene/variant/position/type/aa/supported/numeric/module 표현을 유지하는 대조 후보를 만들고, silent_gene, nonsyn_gene, trunc_gene, unparsed_gene 및 동의 변이만 있는 유전자 비율·동의/비동의가 함께 있는 유전자 비율·미해석 유전자 비율을 추가하는 후보를 비교했습니다. missing은 WT와 구분합니다. split_syn은 동의 변이를 삭제하지 않고 별도 채널에 보존합니다.

cap 보정은 학습 환자 변이 유전자 수의 75% 분위수를 기준으로 초과 행에서만 변이 블록을 (기준/변이량)^power로 줄이며 power 1/0.5를 비교했습니다. 원시 변이 개수·비율·경로 요약은 보존합니다. 기준은 fold 학습에서만 적합합니다.

신규 LR 입력 2,048개, 신규 LightGBM 입력 4,096개입니다. ANOVA와 완전 동일 열 제거는 학습에서만 수행하고 숫자·경로 요약을 보존합니다. 최종 채택 여부와 실제 모델 목록은 model_sizes.json을 보세요. 시험한 전처리가 모두 최종 적용된 것은 아닙니다.

## 검증과 진단
원본 프로필을 그룹으로 묶은 기존 개발 3-fold에서 clean 0.6/희귀 관측 변이 추가 0.3/중복 주석 0.1로 비교했습니다. v6 비중 75% 또는 50%인 제한된 혼합을 시험하고, 선택 점수 +0.002·clean 하락 0.002 이하 기준을 적용했습니다. 통과 후보는 비율을 고정한 뒤 seed=71에서 재점검합니다. 실패하면 v6를 유지합니다. 증강에서 라벨이 유지된다는 가정과 반복 탐색의 선택 편향이 있습니다.

변이량별 진단은 development_burden_report.csv, 암종별 성능은 development_class_report.csv, 많이 헷갈리는 암종 조합은 development_confusions.csv에 있습니다. v6와 v7_candidate 접두사로 구분했습니다. 구간별 환자 수와 암종 구성이 달라 macro F1 수치를 단순 비교하지 않도록 전체 26종 기준과 실제 존재하는 암종 기준을 함께 기록했습니다.

## 산출물과 재현
cencer_v7.ipynb 하나에서 모든 처리·선별·학습·저장을 실행합니다. 프로젝트 Python 파일이나 이전 모델 피클은 필수가 아닙니다. 캐시가 있으면 첫 fold 재학습 일치를 확인한 완료 v6 OOF를 재사용하고, 선택 후 전체 train fit도 재사용할 수 있습니다. 없으면 전부 재학습합니다. 전체 train 캐시는 CV/홀드아웃/재분할에서 금지합니다. 원본 데이터와 requirements-v7.txt 환경은 필요합니다. 최종 모델 artifacts/v7/cancer_model_v7.pkl, 제출 submission/혜진_submission_v7.csv, 변수 전체 목록 artifacts/v7/final_features.csv, 변수별 중요도 final_feature_importance.csv를 생성합니다. 피클은 프로젝트 코드 없는 임시 폴더에서 전체 test 2,546명의 CSV·ID·배치 재현성을 검증했습니다. 모델 크기 {model_path.stat().st_size/1024**2:.2f} MiB입니다. artifacts는 기존 방침대로 Git에서 제외합니다.
"""
(WORK/'analysis_report_v7.md').write_text(report)
(WORK/'requirements-v7.txt').write_text('# Reproduced with '+platform.python_version()+'\n'+'\n'.join(k+'=='+v for k,v in proof['packages'].items())+'\n')
print('Independent replay and report generation passed.',flush=True)

if not accepted7:
 p=WORK/'analysis_report_v7.md'
 p.write_text(p.read_text()+'\n## 채택 실패 시 산출물\n\n신규 후보가 채택되지 않은 경우 주 모델과 제출 파일은 v6를 유지합니다. artifacts/v7/cancer_model_v7.pkl 및 submission_v7.csv는 최종 유지 모델(v6)의 재현 확인용이며 새로운 예측 모델을 뜻하지 않습니다. 동일한 답안지를 새 제출로 배포하지 않습니다.\n')

if not accepted7:
 p=WORK/'analysis_report_v7.md'
 p.write_text(p.read_text().replace('최종 모델 홀드아웃 Macro F1','유지모델(v6) 재현 홀드아웃 Macro F1'))


Independent replay and report generation passed.


In [15]:
importance=[]
for m,w in zip(models,weights):
 if hasattr(m.est,'feature_importances_'):values=m.est.feature_importances_;kind_name='tree_importance'
 elif hasattr(m.est,'coef_'):values=np.max(np.abs(m.est.coef_),axis=0);kind_name='max_absolute_coefficient'
 else:values=np.linalg.norm(m.est.coefs_[0],axis=1);kind_name='first_layer_weight_norm'
 for feature,value in zip(m.names,values):importance.append(dict(model=m.name,view=m.view,feature=feature,importance=float(value),importance_kind=kind_name,ensemble_weight=float(w)))
importance=pd.DataFrame(importance);importance.to_csv(OUT/'final_feature_importance.csv',index=False)
top=importance.sort_values(['model','importance'],ascending=[True,False]).groupby('model').head(10)
top.to_csv(OUT/'top_features_by_model.csv',index=False)
path=WORK/'analysis_report_v7.md'
path.write_text(path.read_text()+'\n## EDA와 최종 변수 해석 파일\n\neda_by_class.csv에는 암종별 환자 수·변이 유전자 수 평균/중앙값/최대값, eda_mutated_gene_burden.csv에는 train/test 기술통계, eda_missing_by_gene.csv에는 유전자별 결측 수가 있습니다. test 통계는 설명에만 사용합니다.\n\nfinal_feature_importance.csv는 실제 최종 변수별 트리 중요도, 최대 절대 선형 계수 또는 신경망 첫 층 가중치 norm입니다. 모델 사이에서 수치 크기를 직접 비교할 수 없으며, 임상적 인과성이나 개별 환자의 기여도를 뜻하지 않습니다. 상위 10개씩은 top_features_by_model.csv에 있습니다.\n')
print('Detailed EDA, feature meanings and fitted feature importance saved.')

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
comparison=pd.read_csv(OUT/'model_comparison.csv',index_col=0)
show=comparison.head(7)
if 'v6_reference' not in show.index:show=pd.concat([show,comparison.loc[['v6_reference']]])
fig,ax=plt.subplots(figsize=(12,6));xx=np.arange(len(show));ax.barh(xx-.18,show.clean,height=.35,label='Clean macro F1');ax.barh(xx+.18,show.noise,height=.35,label='Rare-variant stress macro F1')
ax.set_yticks(xx,show.index);ax.invert_yaxis();ax.set_xlabel('Mean group 3-fold macro F1');ax.legend();fig.tight_layout();fig.savefig(OUT/'validation_comparison_v7.png',dpi=160);plt.close(fig)
path.write_text(path.read_text()+'\n![Development CV comparison](artifacts/v7/validation_comparison_v7.png)\n')

# Rebuild complete gene and final-variable Markdown dictionaries, including the new member.
fams={'gene','variant','position','type','aa','supported','missing','silent_gene','nonsyn_gene','trunc_gene','unparsed_gene'}
selected=importance.copy();selected['gene']=selected.feature.map(lambda f:f.split(':')[1] if ':' in f and f.split(':')[0] in fams else '')
counts=selected[selected.gene.ne('')].groupby('gene').size()
views=selected[selected.gene.ne('')].groupby('gene').view.agg(lambda z:', '.join(sorted(set(z))))
mut=(train[genes].notna()&train[genes].ne('WT'));te=(test[genes].notna()&test[genes].ne('WT'))
gdict=pd.DataFrame({'gene':genes,'train_mutated':mut.sum().values,'train_rate':mut.mean().values,'test_rate':te.mean().values})
gdict['selected_model_feature_uses']=gdict.gene.map(counts).fillna(0).astype(int);gdict['selected_views']=gdict.gene.map(views).fillna('-')
gdict.to_csv(OUT/'gene_dictionary_v7.csv',index=False)
glines=['# v7 전체 유전자 컬럼 사전','','관측 통계이며 유전자 기능/병원성 주석은 원본에 없어 추정하지 않습니다. test_rate는 기술통계로만 사용했습니다. selected_model_feature_uses는 모델 간 중복을 포함합니다. 0개라도 numeric/module 요약에 기여할 수 있습니다.','','| gene | train_mutated | train_rate | test_rate | selected_model_feature_uses | selected_views |','| --- | ---: | ---: | ---: | ---: | --- |']
for r in gdict.itertuples(index=False):glines.append(f'| {r.gene} | {r.train_mutated} | {r.train_rate:.5f} | {r.test_rate:.5f} | {r.selected_model_feature_uses} | {r.selected_views} |')
(WORK/'gene_dictionary_v7.md').write_text('\n'.join(glines)+'\n')
meaning={'silent_gene':'동의 변이가 관측된 유전자','nonsyn_gene':'치환 또는 절단 유형 비동의 변이 유전자 가중값','trunc_gene':'stop/fs/del/ins 유형 관측 여부; 기능 상실 확정 아님','unparsed_gene':'기존 유형으로 해석하지 못한 변이가 있는 유전자','gene':'유전자 변이 존재의 가중값','variant':'정규화한 정확한 단백질 변이','position':'유전자와 단백질 위치','type':'유전자와 변이 유형','aa':'유전자와 아미노산 변화 종류','supported':'학습에서 지지된 비동의 변이 존재','numeric':'변이 부담 로그 또는 변이 유형 비율 요약','module':'명시한 경로 유전자 변이 가중합 log1p','missing':'결측 여부'}
specific={'silent_only_fraction':'동의 변이만 있는 유전자 수 / 변이 유전자 수','mixed_syn_nonsyn_fraction':'동의·비동의 변이가 함께 있는 유전자 수 / 변이 유전자 수','unparsed_gene_fraction':'미해석 변이가 있는 유전자 수 / 변이 유전자 수','log_missing_genes':'log1p(결측 유전자 수); WT와 구분','log_raw_genes':'log1p(변이 유전자 수)','log_effective_genes':'log1p(유전자별 최대 변이 가중치의 합)','log_nonsyn_genes':'log1p(유전자별 비동의 변이 최대 가중치의 합)','log_trunc_genes':'log1p(유전자별 stop/fs/del/ins 최대 가중치의 합)','log_multikind_genes':'log1p(변이 유형이 둘 이상인 유전자 수)','log_mutated_genes':'기존 표현: log1p(변이 유전자 수)','log_events':'기존 표현: log1p(공백으로 분리한 전체 변이 토큰 수; 중복 포함)','multi_event_excess':'기존 표현: log1p(max(변이 토큰 수-변이 유전자 수,0))','log_mean_position':'기존 표현: log1p(변이 위치 숫자 평균)'}
def feature_meaning(f):
 parts=f.split(':');family=parts[0];tail=parts[-1]
 if family=='numeric':
  if tail in specific:return specific[tail]
  if tail.startswith('gene_fraction_'):return '해당 유형의 유전자별 최대 가중치 합 / 변이 유전자 수; 유형 간 중복 가능'
  if tail.startswith('fraction_'):return '기존 표현: 해당 유형 토큰 수 / 전체 변이 토큰 수'
  if tail.startswith('log_'):return '기존 표현: log1p(해당 유형 변이 토큰 수)'
 return meaning.get(family,'표현 정의 참조')
slines=['# v7 최종 학습 변수 전체 목록','','모델별 실제 입력 목록입니다. 같은 변수도 view와 모델에 따라 의미/가중치가 다릅니다. 선정 근거는 고정 v6 대조군의 기존 선별, 또는 v7 학습 내 ANOVA/완전 동일 열 제거입니다. 숫자·경로 요약은 v7에서 항상 보존했습니다. 임상적 인과성은 확인하지 않았습니다. 중요도는 모델 안에서만 비교하며 서로 다른 지표를 비교할 수 없습니다.','','| model | view | feature | meaning | importance | metric |','| --- | --- | --- | --- | ---: | --- |']
for r in selected.itertuples(index=False):slines.append(f'| {r.model} | {r.view} | {r.feature} | {feature_meaning(r.feature)} | {r.importance:.7g} | {r.importance_kind} |')
(WORK/'selected_variables_v7.md').write_text('\n'.join(slines)+'\n')


Detailed EDA, feature meanings and fitted feature importance saved.


Matplotlib is building the font cache; this may take a moment.


## 원본 프로필의 정답 모호성
동일한 유전자 정보에 여러 암종이 붙은 경우를 기술통계로 확인합니다. 프로필로 정답을 복사하는 학습이나 테스트 보정에는 사용하지 않습니다.

In [16]:
# Descriptive ambiguity audit; never used as a profile-to-label lookup.
profile_counts=pd.crosstab(groups,y)
profile_rows=profile_counts.sum(axis=1)
conflicting=profile_counts[(profile_counts>0).sum(axis=1)>1]
ambiguity=dict(unique_profiles=len(profile_counts),conflicting_profiles=len(conflicting),rows_in_conflicting_profiles=int(conflicting.sum().sum()),minimum_training_errors_given_raw_features=int((profile_rows-profile_counts.max(axis=1)).sum()),descriptive_training_accuracy_ceiling=float(profile_counts.max(axis=1).sum()/len(train)))
(OUT/'raw_profile_label_ambiguity.json').write_text(json.dumps(ambiguity,indent=2))
print('Identical original profiles with different labels:',ambiguity['conflicting_profiles'],'groups,',ambiguity['rows_in_conflicting_profiles'],'rows')
print('This is a descriptive training-data limit, not an estimate of test performance.')

path=WORK/'analysis_report_v7.md'
text=path.read_text()
marker='\n## 원본 프로필 정답 모호성\n'
text=text.split(marker)[0]+marker+f"\n동일 원본 변이 프로필인데 암종이 다른 그룹 {ambiguity['conflicting_profiles']}개, 해당 행 {ambiguity['rows_in_conflicting_profiles']}개입니다. 프로필을 함께 묶어 검증에서 분리했습니다. 정답 복사 규칙으로 사용하지 않았으며 test 성능 추정치가 아닙니다.\n"
path.write_text(text)


DATA /Users/mac/project/cancer_project/data OUTPUT /Users/mac/project/cancer_project/artifacts/v7 Train/test (6201, 4386) (2546, 4385) Dev/holdout 4960 1241
Identical original profiles with different labels: 447 groups, 1005 rows
This is a descriptive training-data limit, not an estimate of test performance.


## Git에서도 읽을 수 있는 실험·진단 요약
전체 CSV 없이도 결과를 확인하도록 분석 보고서에 핵심 표를 포함합니다.

In [17]:
def compact_markdown_table(frame):
 cols=[str(x) for x in frame.columns]
 rows=['| '+' | '.join(cols)+' |','| '+' | '.join(['---']*len(cols))+' |']
 for row in frame.itertuples(index=False,name=None):
  values=[f'{x:.4f}' if isinstance(x,(float,np.floating)) else str(x) for x in row]
  rows.append('| '+' | '.join(values)+' |')
 return '\n'.join(rows)
comparison=pd.read_csv(OUT/'model_comparison.csv',index_col=0)
new_names=list(json.loads((OUT/'configurations_v7.json').read_text()))
individual=comparison.loc[new_names,['clean','noise','score']].sort_values('score',ascending=False).reset_index(names='candidate')
burden_old=pd.read_csv(OUT/'v6_development_burden_report.csv')
burden_new=pd.read_csv(OUT/'v7_candidate_development_burden_report.csv')
burden=burden_old[['burden','n','present_classes','accuracy']].rename(columns={'accuracy':'v6_accuracy'})
burden['candidate_accuracy']=burden_new.accuracy
confusions=pd.read_csv(OUT/'v6_development_confusions.csv').head(10)
section='\n## 전처리별 단독 CV 결과\n\n'+compact_markdown_table(individual)
section+='\n\n단독 후보를 v6 앙상블과 직접 비교한 표입니다. 실제 채택 판단에는 미리 정한 v6 혼합 후보와 재분할 점수를 사용했습니다. 표기 정합성 개선만으로 성능 향상을 보장하지 않습니다.\n'
section+='\n## 변이량별 개발 OOF 정확도\n\n'+compact_markdown_table(burden)
section+='\n\n후보는 1차 CV에서 고른 고정 후보이며 미채택일 수도 있습니다. 구간별 환자와 암종 구성이 달라 구간 간 점수 차이를 변이량의 인과효과로 해석하지 않습니다.\n'
section+='\n## v6의 주요 오분류 조합\n\n'+compact_markdown_table(confusions)+'\n'
p=WORK/'analysis_report_v7.md';body=p.read_text();marker='\n## 전처리별 단독 CV 결과\n';p.write_text(body.split(marker)[0]+section)
print('Readable preprocessing, burden and confusion tables added to report.')


Readable preprocessing, burden and confusion tables added to report.
